In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict , Any , Tuple
from sklearn.metrics.pairwise import cosine_similarity
import os
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

Data Ingestion

In [ ]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- 1. Load Documents ---
def load_pdfs_from_dir(directory_path: str):
    print(f"Loading PDFs from: {directory_path}")
    if not os.path.exists(directory_path):
        # Fallback to current directory if ../data/pdf isn't found
        print(f"Warning: {directory_path} not found. Checking current directory...")
        directory_path = "./" 

    dir_loader = DirectoryLoader(
        directory_path,
        glob="**/*.pdf",
        loader_cls=PyMuPDFLoader,
        show_progress=True
    )
    docs = dir_loader.load()
    return docs

# --- 2. Split Documents (The missing function) ---
def split_documents(docs):
    print(f"Splitting {len(docs)} pages into chunks...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100,
        length_function=len,
        add_start_index=True,
    )
    chunks = text_splitter.split_documents(docs)
    print(f"Created {len(chunks)} chunks.")
    return chunks

# --- 3. Main Execution ---
if __name__ == "__main__":
    # Ensure these paths exist on your machine
    PDF_DIRECTORY = "../data/pdf" 
    PERSIST_DIR = "./data/vector_store"
    
    # Create the vector store directory if it doesn't exist
    if not os.path.exists(PERSIST_DIR):
        os.makedirs(PERSIST_DIR)

    # A. Load
    try:
        raw_documents = load_pdfs_from_dir(PDF_DIRECTORY)
        if not raw_documents:
            print("No PDFs found! Please check your directory path.")
        else:
            # B. Split
            chunks = split_documents(raw_documents)

            # C. Initialize Hugging Face Embeddings
            print("Initializing Hugging Face Embeddings (all-MiniLM-L6-v2)...")
            hf_embeddings = HuggingFaceEmbeddings(
                model_name="all-MiniLM-L6-v2",
                model_kwargs={'device': 'cpu'}
            )

            # D. Store in Chroma
            print(f"Vectorizing {len(chunks)} chunks and saving to {PERSIST_DIR}...")
            v_store = Chroma.from_documents(
                documents=chunks,
                embedding=hf_embeddings,
                persist_directory=PERSIST_DIR,
                collection_name="pdf_documents"
            )
            
            # Explicitly persist (Good practice for local DBs)
            print("\n All PDFs indexed successfully!")
            
    except Exception as e:
        print(f"An error occurred: {e}")

Loading PDFs from: ../data/pdf


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  9.21it/s]

Splitting 26 pages into chunks...
Created 74 chunks.
Initializing Hugging Face Embeddings (all-MiniLM-L6-v2)...



/tmp/ipykernel_568259/3862232633.py:58: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  hf_embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vectorizing 74 chunks and saving to ./data/vector_store...

🚀 All PDFs indexed successfully!


Retrival

In [6]:
import os
from openai import OpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

class OpenRouterLLM:
    """A custom wrapper to make the OpenAI client work with LangChain."""
    def __init__(self, api_key):
        self.client = OpenAI(
            api_key=api_key,
            base_url="https://openrouter.ai/api/v1"
        )
        self.model = "meta-llama/llama-3-8b-instruct"

    def invoke(self, input_text):
        # This matches the specific OpenRouter logic you provided
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": str(input_text)}]
        )
        return response.choices[0].message.content

class RAGPipeline:
    def __init__(self, api_key, persist_directory="./data/vector_store"):
        # 1. Initialize local embeddings (Must match your ingestion)
        self.embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        
        # 2. Connect to ChromaDB
        self.vectorstore = Chroma(
            persist_directory=persist_directory,
            embedding_function=self.embeddings,
            collection_name="pdf_documents"
        )
        
        # 3. Initialize your OpenRouter Client
        self.llm = OpenRouterLLM(api_key=api_key)

    def get_answer(self, question):
        # 4. Retrieval
        retriever = self.vectorstore.as_retriever(search_kwargs={"k": 3})
        context_docs = retriever.invoke(question)
        
        # Format the snippets into one string
        context_text = "\n\n".join([doc.page_content for doc in context_docs])
        
        # 5. Build the final prompt
        full_prompt = f"""
        Answer the question based ONLY on the following context:
        {context_text}
        
        Question: {question}
        Answer:
        """
        
        # 6. Generate via OpenRouter
        return self.llm.invoke(full_prompt)

# --- Execution ---
if __name__ == "__main__":
    # Your specific key
    API_KEY = "Enter API key"
    
    rag = RAGPipeline(api_key=API_KEY)
    
    user_query = "How is performance evaluation done?"
    print("\n--- Querying OpenRouter ---")
    print(rag.get_answer(user_query))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_568259/2790999473.py:31: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self.vectorstore = Chroma(



--- Querying OpenRouter ---
According to the context, performance evaluation is done by:

1. Determining what needs to be monitored and measured, including information security processes and controls.
2. Selecting methods for monitoring, measurement, analysis, and evaluation to ensure valid results.
3. Deciding when the monitoring and measuring shall be performed.
4. Identifying who shall monitor and measure.
5. Deciding when the results from monitoring and measurement shall be analyzed and evaluated.
6. Identifying who shall analyze and evaluate these results.

Documented information shall be available as evidence of the results.


Corrective RAG

In [ ]:
from openai import OpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

class CorrectiveConversationalRAG:
    def __init__(self, api_key):
        # 1. OpenRouter Client Setup
        self.client = OpenAI(
            api_key=api_key,
            base_url="https://openrouter.ai/api/v1"
        )
        self.model = "meta-llama/llama-3-8b-instruct"
        
        self.chat_history = [] 
        
        # 2. Local Embeddings (HuggingFace)
        # This will download the model (~80MB) on first run
        self.embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        
        # 3. Vector Store Connection
        # Ensure this path matches where you indexed your PDFs
        self.vectorstore = Chroma(
            persist_directory="./data/vector_store",
            embedding_function=self.embeddings,
            collection_name="pdf_documents"
        )

    def grade_documents(self, question, documents):
        """Self-Correction: Determines if retrieved docs actually help."""
        if not documents:
            return "NO"
            
        context = "\n".join([doc.page_content for doc in documents])
        grading_prompt = f"Is this context relevant to the question: '{question}'? Answer ONLY YES or NO.\n\nContext: {context}"
        
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": grading_prompt}]
            )
            return response.choices[0].message.content.strip().upper()
        except Exception as e:
            print(f"Grading Error: {e}")
            return "YES" # Fallback to continue even if grader fails

    def get_answer(self, user_input):
        # A. Retrieval
        retriever = self.vectorstore.as_retriever(search_kwargs={"k": 3})
        docs = retriever.invoke(user_input)
        
        # B. Corrective Check
        grade = self.grade_documents(user_input, docs)
        
        if "YES" in grade:
            context_text = "\n\n".join([d.page_content for d in docs])
        else:
            context_text = "Note: No direct matches found in documents. Answering from general knowledge."

        # C. Construct Prompt with Memory
        memory_str = "\n".join(self.chat_history[-4:]) # Last 2 exchanges
        
        full_prompt = f"""
        Previous Conversation:
        {memory_str}

        Context from PDFs:
        {context_text}
        
        Question: {user_input}
        Answer:"""

        # D. Generate Final Answer
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": full_prompt}]
            )
            answer = response.choices[0].message.content
            
            # E. Update Memory
            self.chat_history.append(f"User: {user_input}")
            self.chat_history.append(f"AI: {answer}")
            return answer
        except Exception as e:
            return f"Authentication or API Error: {str(e)}"

# --- Main ---
if __name__ == "__main__":
    # REPLACING THE KEY: Use a new key from OpenRouter
    MY_KEY = "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"
    
    # Quick sanity check for the user
    if "sk-or" not in MY_KEY:
        print("Error: Please use a valid OpenRouter key starting with 'sk-or-v1-'")
    else:
        rag_system = CorrectiveConversationalRAG(api_key=MY_KEY)
        print("\n--- Conversational CRAG Active (Type 'exit' to quit) ---")
        
        while True:
            u_input = input("\nYou: ")
            if u_input.lower() in ["exit", "quit", "q"]:
                break
            print("\nAI is thinking...")
            print(f"AI: {rag_system.get_answer(u_input)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Conversational CRAG Active (Type 'exit' to quit) ---

AI is thinking...
Grading Error: Error code: 401 - {'error': {'message': 'User not found.', 'code': 401}}
AI: Authentication or API Error: Error code: 401 - {'error': {'message': 'User not found.', 'code': 401}}
